<a href="https://colab.research.google.com/github/RashiBista/GraphRAG/blob/main/02_vector_baseline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [15]:
!pip -q install sentence-transformers

In [16]:
import json
import numpy as np
from pathlib import Path
from google.colab import drive

drive.mount("/content/drive")
WORKDIR = Path("/content/drive/MyDrive/HybridGraphRAG/twowiki")

def load_jsonl(path):
    with path.open("r", encoding="utf-8") as f:
        return [json.loads(line) for line in f]

chunks = load_jsonl(WORKDIR / "document_chunks.jsonl")
questions = load_jsonl(WORKDIR / "questions.jsonl")
print(len(chunks), "chunks,", len(questions), "questions")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
182 chunks, 20 questions


In [17]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")
emb = model.encode(
    [c["text"] for c in chunks],
    normalize_embeddings=True,
    show_progress_bar=True,
)

print("Embedding matrix shape: ", emb.shape)
np.save(WORKDIR / "emb.npy", emb)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Batches:   0%|          | 0/6 [00:00<?, ?it/s]

Embedding matrix shape:  (182, 384)


In [18]:
def retrieve(question, k=5):
    q_emb = model.encode([question], normalize_embeddings=True)[0]
    scores = emb @ q_emb
    top = np.argsort(-scores)[:k]
    return [(int(i), float(scores[i])) for i in top]

def show(q, k=5):
    print(f"\n[{q['type']}] {q['question']}")
    print("Answer:", q["answer"], "| gold paragraphs:", q["gold_titles"])
    for rank, (i, s) in enumerate(retrieve(q["question"], k), 1):
        title = chunks[i]["title"]
        mark = "ok" if title in q["gold_titles"] else "  "
        print(f"  {rank}. {mark} {s:.3f}  {title}")

In [19]:
first_of_type = {}
for q in questions:
    first_of_type.setdefault(q["type"], q)

for q in first_of_type.values():
    show(q, k=5)


[bridge_comparison] Which film has the director who was born first, Slunce, Seno, Jahody or To Sleep With Anger?
Answer: To Sleep With Anger | gold paragraphs: ['Charles Burnett (director)', 'Slunce, seno, jahody', 'To Sleep with Anger', 'Zdeněk Troška']
  1. ok 0.648  Slunce, seno, jahody
  2.    0.577  Slunce, seno, erotika
  3. ok 0.569  To Sleep with Anger
  4.    0.507  Marie Pilátová
  5.    0.486  Slunce, seno a pár facek

[comparison] Who was born earlier, Geoff Bagnall or Caroline Huppert?
Answer: Caroline Huppert | gold paragraphs: ['Caroline Huppert', 'Geoff Bagnall']
  1. ok 0.589  Geoff Bagnall
  2. ok 0.544  Caroline Huppert
  3.    0.379  Tom Dickinson
  4.    0.364  Caroline Robbins
  5.    0.335  Ben Palmer

[compositional] Where was the place of death of Naïma Ben Ali's husband?
Answer: Jeddah | gold paragraphs: ['Naïma Ben Ali', 'Zine El Abidine Ben Ali']
  1. ok 0.631  Naïma Ben Ali
  2.    0.485  Leïla Ben Ali
  3.    0.471  Ibrahim Ben Ali
  4.    0.444  Belhasse

In [20]:
title_to_idx = {c["title"]: i for i, c in enumerate(chunks)}

def gold_ranks(q):
    q_emb = model.encode([q["question"]], normalize_embeddings=True)[0]
    order = np.argsort(-(emb @ q_emb))
    rank_of = {int(idx): r for r, idx in enumerate(order, start=1)}
    return {t: rank_of[title_to_idx[t]] for t in q["gold_titles"]}

rows = [(q, gold_ranks(q)) for q in questions]
print("Done:", len(rows), "questions")

Done: 20 questions


In [21]:
KS = [2, 5, 10]

def summarize(rows, ks=KS):
    types = sorted({q["type"] for q, _ in rows})
    print(f"{'type':<18}{'n':>3}  " + "   ".join(f"k={k:<8}" for k in ks))
    for t in types + ["ALL"]:
        sub = [(q, r) for q, r in rows if t == "ALL" or q["type"] == t]
        cells = []
        for k in ks:
            rec = np.mean([sum(rk <= k for rk in r.values()) / len(r) for _, r in sub])
            allf = sum(all(rk <= k for rk in r.values()) for _, r in sub)
            cells.append(f"{rec:.2f} ({allf}/{len(sub)})")
        print(f"{t:<18}{len(sub):>3}  " + "   ".join(f"{c:<10}" for c in cells))

summarize(rows)

print("\nGold paragraphs ranked below 5:")
for q, r in rows:
    missed = {t: rk for t, rk in r.items() if rk > 5}
    if missed:
        print(f"[{q['type']}] {q['question'][:70]}")
        for t, rk in missed.items():
            print(f"     rank {rk:>3}  {t}")

type                n  k=2          k=5          k=10      
bridge_comparison   5  0.40 (0/5)   0.60 (0/5)   0.75 (1/5)
comparison          5  1.00 (5/5)   1.00 (5/5)   1.00 (5/5)
compositional       5  0.50 (0/5)   0.60 (1/5)   0.80 (3/5)
inference           5  0.50 (0/5)   0.70 (2/5)   0.90 (4/5)
ALL                20  0.60 (5/20)   0.72 (8/20)   0.86 (13/20)

Gold paragraphs ranked below 5:
[bridge_comparison] Which film has the director who was born first, Slunce, Seno, Jahody o
     rank   6  Charles Burnett (director)
     rank   7  Zdeněk Troška
[bridge_comparison] Which film has the director who died earlier, The Glass Menagerie (198
     rank  20  Karl Anton
     rank  36  Paul Newman
[bridge_comparison] Which film whose director is younger, The Devil'S Pass Key or La Bella
     rank  58  Erich von Stroheim
[bridge_comparison] Which film has the director who died earlier, Everything'S Ducky or Ka
     rank  44  Don Taylor (American actor and director)
     rank   6  M. Krishna

In [35]:
import re, string
from collections import Counter
import unicodedata

def normalize_answer(s):
    s = unicodedata.normalize("NFKC", s).lower()
    s = "".join(ch for ch in s if not unicodedata.category(ch).startswith("P"))
    s = re.sub(r"\b(a|an|the)\b", " ", s)
    return " ".join(s.split())

def exact_match(pred, gold):
    return float(normalize_answer(pred) == normalize_answer(gold))

def f1_score(pred, gold):
    p, g = normalize_answer(pred).split(), normalize_answer(gold).split()
    n = sum((Counter(p) & Counter(g)).values())
    if n == 0:
        return 0.0
    precision, recall = n / len(p), n / len(g)
    return 2 * precision * recall / (precision + recall)


baseline k=5   EM 0.30   F1 0.37
baseline k=10  EM 0.50   F1 0.59
oracle         EM 0.70   F1 0.86


In [23]:
!pip -q install groq

In [30]:

from groq import Groq
from google.colab import userdata

groq_client = Groq(api_key=userdata.get("GROQ_API_KEY"))
LLM = "openai/gpt-oss-20b"   # NOT the safeguard model used earlier

def answer_question(question, paragraphs):
    context = "\n\n".join(f"[{i+1}] {p}" for i, p in enumerate(paragraphs))
    prompt = f"""Answer the question using only the paragraphs below.
Give just the answer (a name, place, date, or yes/no) with no explanation.
If the paragraphs do not contain enough information, answer exactly: unknown

PARAGRAPHS:
{context}

QUESTION: {question}
ANSWER:"""
    r = groq_client.chat.completions.create(
        model=LLM,
        messages=[{"role": "user", "content": prompt}],
        temperature=0,
    )
    return r.choices[0].message.content.strip()

In [31]:
import time

def run_baseline(k):
    results = []
    for q in questions:
        paras = [chunks[i]["text"] for i, _ in retrieve(q["question"], k)]
        pred = answer_question(q["question"], paras)
        results.append({"type": q["type"], "q": q["question"], "gold": q["answer"],
                        "pred": pred,
                        "em": exact_match(pred, q["answer"]),
                        "f1": f1_score(pred, q["answer"])})
        time.sleep(1)   # gentle on rate limits
    return results

baseline = {k: run_baseline(k) for k in (5, 10)}

for k, res in baseline.items():
    print(f"\n=== k={k} ===")
    for t in sorted({r['type'] for r in res}) + ["ALL"]:
        sub = [r for r in res if t == "ALL" or r["type"] == t]
        print(f"{t:<18} EM {np.mean([r['em'] for r in sub]):.2f}   F1 {np.mean([r['f1'] for r in sub]):.2f}")

print("\nWrong answers at k=5:")
for r in baseline[5]:
    if r["em"] < 1:
        print(f"[{r['type']}] gold={r['gold']!r} | pred={r['pred']!r}")


=== k=5 ===
bridge_comparison  EM 0.00   F1 0.00
comparison         EM 1.00   F1 1.00
compositional      EM 0.00   F1 0.13
inference          EM 0.20   F1 0.36
ALL                EM 0.30   F1 0.37

=== k=10 ===
bridge_comparison  EM 0.20   F1 0.20
comparison         EM 1.00   F1 1.00
compositional      EM 0.20   F1 0.43
inference          EM 0.40   F1 0.66
ALL                EM 0.45   F1 0.57

Wrong answers at k=5:
[bridge_comparison] gold='To Sleep With Anger' | pred='unknown'
[bridge_comparison] gold='Monsieur Sans-Gêne' | pred='unknown'
[bridge_comparison] gold='La Bella Di Lodi' | pred='unknown'
[bridge_comparison] gold="Everything'S Ducky" | pred='unknown'
[bridge_comparison] gold='The Big Show-Off' | pred='unknown'
[compositional] gold='Jeddah' | pred='unknown'
[compositional] gold='Czech Republic' | pred='Czech'
[compositional] gold='Cahiers du cinéma' | pred='unknown'
[compositional] gold='Young Australian of the Year' | pred='unknown'
[compositional] gold='Rome' | pred='unkno

In [33]:
import json

title_to_text = {c["title"]: c["text"] for c in chunks}

oracle = []
for q in questions:
    paras = [title_to_text[t] for t in q["gold_titles"]]
    pred = answer_question(q["question"], paras)
    oracle.append({"type": q["type"], "q": q["question"], "gold": q["answer"],
                   "pred": pred,
                   "em": exact_match(pred, q["answer"]),
                   "f1": f1_score(pred, q["answer"])})
    time.sleep(1)

print("=== ORACLE (gold paragraphs only) ===")
for t in sorted({r["type"] for r in oracle}) + ["ALL"]:
    sub = [r for r in oracle if t == "ALL" or r["type"] == t]
    print(f"{t:<18} EM {np.mean([r['em'] for r in sub]):.2f}   F1 {np.mean([r['f1'] for r in sub]):.2f}")

print("\nOracle wrong answers:")
for r in oracle:
    if r["em"] < 1:
        print(f"[{r['type']}] gold={r['gold']!r} | pred={r['pred']!r}")

print("\nWrong answers at k=10:")
for r in baseline[10]:
    if r["em"] < 1:
        print(f"[{r['type']}] gold={r['gold']!r} | pred={r['pred']!r}")

results = {"baseline_k5": baseline[5], "baseline_k10": baseline[10], "oracle": oracle}
with open(WORKDIR / "baseline_results.json", "w", encoding="utf-8") as f:
    json.dump(results, f, ensure_ascii=False, indent=2)
print("\nSaved to", WORKDIR / "baseline_results.json")

=== ORACLE (gold paragraphs only) ===
bridge_comparison  EM 1.00   F1 1.00
comparison         EM 1.00   F1 1.00
compositional      EM 0.40   F1 0.81
inference          EM 0.20   F1 0.60
ALL                EM 0.65   F1 0.85

Oracle wrong answers:
[compositional] gold='Jeddah' | pred='Jeddah, Saudi Arabia'
[compositional] gold='Czech Republic' | pred='Czech'
[compositional] gold='Young Australian of the Year' | pred='Young Australian of the Year Award'
[inference] gold='Thyra' | pred='Thyra Dannebod'
[inference] gold='Wilhelmine Amalia of Brunswick-Lüneburg' | pred='Wilhelmine Amalia of Brunswick‑Lüneburg'
[inference] gold='Maria Feodorovna' | pred='Empress Maria Feodorovna of Russia'
[inference] gold='Hernando Pizarro' | pred='unknown'

Wrong answers at k=10:
[bridge_comparison] gold='Monsieur Sans-Gêne' | pred='unknown'
[bridge_comparison] gold='La Bella Di Lodi' | pred='unknown'
[bridge_comparison] gold="Everything'S Ducky" | pred='unknown'
[bridge_comparison] gold='The Big Show-Off' 

In [36]:
def rescore(res):
    for r in res:
        r["em"] = exact_match(r["pred"], r["gold"])
        r["f1"] = f1_score(r["pred"], r["gold"])
    return res

for name, res in [("baseline k=5", baseline[5]), ("baseline k=10", baseline[10]), ("oracle", oracle)]:
    rescore(res)
    print(f"{name:<14} EM {np.mean([r['em'] for r in res]):.2f}   F1 {np.mean([r['f1'] for r in res]):.2f}")

baseline k=5   EM 0.30   F1 0.37
baseline k=10  EM 0.50   F1 0.59
oracle         EM 0.70   F1 0.86


In [38]:
from groq import RateLimitError

def chat_with_retry(**kwargs):
    for attempt in range(8):
        try:
            return groq_client.chat.completions.create(**kwargs)
        except RateLimitError:
            wait = 3 * (attempt + 1)
            print(f"  rate limited, waiting {wait}s...")
            time.sleep(wait)
    raise RuntimeError("Still rate limited after 8 retries")

def answer_question(question, paragraphs):
    context = "\n\n".join(f"[{i+1}] {p}" for i, p in enumerate(paragraphs))
    prompt = f"""Answer the question using only the paragraphs below.
Give only the answer, copied as it is written in the paragraphs: a name, place, date, or yes/no. Do not add titles, countries, or other extra words, and do not explain.
If the paragraphs do not contain enough information, answer exactly: unknown

PARAGRAPHS:
{context}

QUESTION: {question}
ANSWER:"""
    r = chat_with_retry(
        model=LLM,
        messages=[{"role": "user", "content": prompt}],
        temperature=0,
    )
    return r.choices[0].message.content.strip()

def summarize_run(name, res):
    print(f"\n=== {name} ===")
    for t in sorted({r["type"] for r in res}) + ["ALL"]:
        sub = [r for r in res if t == "ALL" or r["type"] == t]
        print(f"{t:<18} EM {np.mean([r['em'] for r in sub]):.2f}   F1 {np.mean([r['f1'] for r in sub]):.2f}")

# safety net: only recompute the baseline if the runtime was restarted
if "baseline" not in globals():
    baseline = {k: run_baseline(k) for k in (5, 10)}

oracle = run_oracle()

summarize_run("baseline k=5", baseline[5])
summarize_run("baseline k=10", baseline[10])
summarize_run("oracle", oracle)

print("\nOracle wrong answers:")
for r in oracle:
    if r["em"] < 1:
        print(f"[{r['type']}] {r['q']}\n   gold={r['gold']!r} | pred={r['pred']!r}")

with open(WORKDIR / "baseline_results.json", "w", encoding="utf-8") as f:
    json.dump({"baseline_k5": baseline[5], "baseline_k10": baseline[10], "oracle": oracle},
              f, ensure_ascii=False, indent=2)
print("\nSaved.")


=== baseline k=5 ===
bridge_comparison  EM 0.00   F1 0.00
comparison         EM 1.00   F1 1.00
compositional      EM 0.20   F1 0.20
inference          EM 0.20   F1 0.27
ALL                EM 0.35   F1 0.37

=== baseline k=10 ===
bridge_comparison  EM 0.20   F1 0.20
comparison         EM 0.80   F1 0.96
compositional      EM 0.40   F1 0.50
inference          EM 0.40   F1 0.65
ALL                EM 0.45   F1 0.58

=== oracle ===
bridge_comparison  EM 0.80   F1 0.88
comparison         EM 1.00   F1 1.00
compositional      EM 0.60   F1 0.88
inference          EM 0.40   F1 0.78
ALL                EM 0.70   F1 0.88

Oracle wrong answers:
[bridge_comparison] Which film has the director who died earlier, The Glass Menagerie (1987 Film) or Monsieur Sans-Gêne?
   gold='Monsieur Sans-Gêne' | pred='Monsieur Sans- Gêne'
[compositional] Where was the place of death of Naïma Ben Ali's husband?
   gold='Jeddah' | pred='Jeddah, Saudi Arabia'
[compositional] What is the award that the director of film Fo